In [2]:
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel, branch
from dataclasses import Field

from  langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel
from typing import Literal, TypedDict
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END
from IPython.display import Image,display, Markdown



# This function will load all the variables from the .env file and will
# make them available in the os.environ dictionary (env variables)
load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()
class graph_schema(TypedDict):
    name:str
    message:str

def welcome(state:graph_schema)->graph_schema:
    curr_name=state['name']
    curr_message=state['message']
    curr_respone=llm_openai.invoke(f"My name is {curr_name}.{curr_message}").content
    state['message']=f"{curr_message}. LLM response {curr_respone}"
    return state

graph=StateGraph(graph_schema)

graph.add_node("welcome",welcome)

graph.add_edge(START,"welcome")
graph.add_edge("welcome",END)
f_graph=graph.compile()

# Display Mermaid text instead of trying to render as PNG
print(f_graph.get_graph().draw_mermaid())

f_graph.invoke({"name":"ujjwal","message":"How are you?"})

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	welcome(welcome)
	__end__([<p>__end__</p>]):::last
	__start__ --> welcome;
	welcome --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



{'name': 'ujjwal',
 'message': 'How are you?. LLM response Hi Ujjwal! I’m doing great, thanks for asking. How about you? Anything interesting on your mind today?'}